# Buckley–Leverett Waterflood Simulator

A 1-D two-phase (oil + water) flow simulator based on the **Buckley–Leverett** equation.
Explore how the water saturation front propagates through a porous medium and how it responds
to the viscosity ratio between water and oil.

> **Interactive Matplotlib script:** `python buckley_leverett.py` launches a full 3-D dashboard
> with real-time sliders for time, position, and water viscosity.
>
> **This notebook** covers the theory, step-by-step code, publication-quality static plots,
> and an `ipywidgets` interactive explorer.

---
### Interactive Script Preview

![Dashboard](screenshots/dashboard.png)

*Full 3-D saturation surface with time-slice (cyan) and space-slice (orange) planes,*
*2-D profile, saturation history, and fractional-flow panel.*


---
## Mathematical Background

### Governing Equation

The Buckley–Leverett equation describes 1-D transport of water in an incompressible, immiscible two-phase system:

$$\frac{\partial S_w}{\partial t} + u \frac{\partial f_w}{\partial x} = 0$$

where $S_w(x,t)$ is the water saturation, $u$ is the (constant) total Darcy velocity,
and $f_w$ is the **fractional flow** of water.

### Fractional Flow

$$f_w = \frac{\lambda_w}{\lambda_w + \lambda_o} = \frac{k_{rw}/\mu_w}{k_{rw}/\mu_w + k_{ro}/\mu_o}$$

### Corey Relative Permeabilities

$$k_{rw}(S_w) = S_n^{N_w}, \qquad k_{ro}(S_w) = (1-S_n)^{N_o}, \qquad S_n = \frac{S_w - S_{wc}}{1 - S_{wc} - S_{or}}$$

### Shock Front — Welge Construction

Because $f_w(S_w)$ is S-shaped, a **shock discontinuity** forms at the leading edge of the flood.
The shock saturation $S_w^*$ and speed are found by the **Welge tangent**:
draw the tangent from $(S_{wc},\,0)$ to the $f_w$ curve.
The tangent point gives $S_w^*$ and the slope gives:

$$v_s = \frac{f_w(S_w^*) - 0}{S_w^* - S_{wc}} = \frac{df_w}{dS_w}\bigg|_{S_w^*}$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# ── global dark theme ────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#111111',
    'axes.facecolor':   '#1a1a1a',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      'white',
    'ytick.color':      'white',
    'axes.edgecolor':   '#555555',
})

CMAP = plt.cm.plasma    # saturation colour map used throughout

def _dark(ax):
    """Apply dark-theme styling to a matplotlib Axes."""
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    for sp in ax.spines.values():
        sp.set_color('#555555')

---
## Model Parameters

| Parameter | Symbol | Default | Description |
|-----------|--------|---------|-------------|
| Connate water saturation | $S_{wc}$ | 0.20 | Irreducible water saturation |
| Residual oil saturation  | $S_{or}$ | 0.20 | Irreducible oil saturation |
| Oil viscosity            | $\mu_o$  | 2.0 cP | Fixed reference viscosity |
| Water viscosity (slider) | $\mu_w$  | 0.2 – 10 cP | Varies in the interactive section |
| Corey exponent (water)   | $N_w$    | 2.0 | Controls $k_{rw}$ concavity |
| Corey exponent (oil)     | $N_o$    | 2.0 | Controls $k_{ro}$ concavity |


In [ ]:
# ── Rock / fluid parameters ───────────────────────────────────────────────────
S_WC  = 0.20          # connate water saturation
S_OR  = 0.20          # residual oil saturation
MU_O  = 2.0           # oil viscosity  [cP]
N_W   = 2.0           # Corey exponent – water
N_O   = 2.0           # Corey exponent – oil
U_TOT = 1.0           # total Darcy velocity (normalised)

def normalised_sat(sw):
    """Effective (normalised) water saturation in [0, 1]."""
    return np.clip((sw - S_WC) / (1.0 - S_WC - S_OR), 0.0, 1.0)

def rel_perms(sw):
    """Corey relative permeabilities (krw, kro)."""
    sn = normalised_sat(sw)
    return sn ** N_W, (1.0 - sn) ** N_O

def fractional_flow(sw, mu_w):
    """Water fractional flow  f_w(S_w ; mu_w)."""
    krw, kro = rel_perms(sw)
    lam_w = krw / mu_w
    lam_o = kro / MU_O
    denom = lam_w + lam_o
    return np.where(denom > 1e-12, lam_w / denom, 0.0)

def shock_saturation(mu_w, n_pts=500):
    """Welge tangent construction: returns shock-front saturation S_w*."""
    sw_arr = np.linspace(S_WC + 1e-4, 1.0 - S_OR - 1e-4, n_pts)
    fw_arr = fractional_flow(sw_arr, mu_w)
    slope  = fw_arr / (sw_arr - S_WC)    # slope from (S_wc, 0)
    return sw_arr[np.argmax(slope)]

---
## Numerical Solver

We use an **explicit upwind finite-difference** scheme on a 1-D uniform grid of $N_x = 200$ cells:

$$S_i^{n+1} = S_i^n - \frac{u\,\Delta t}{\Delta x}\left(f_w(S_i^n) - f_w(S_{i-1}^n)\right)$$

**Boundary conditions:**
- Left inlet: $S = 1 - S_{or}$ (full water injection)
- Right outlet: outflow (no explicit condition)

**Stability:** the time step satisfies $\Delta t \leq 0.45\,\Delta x / (u\,\max|f_w'|)$.
Stored snapshots are sub-cycled so the output spacing is exactly $T_{\max}/N_t$.


In [ ]:
NX   = 200          # spatial cells
NT   = 300          # time snapshots stored
XMAX = 1.0          # dimensionless reservoir length
TMAX = 1.0          # pore-volumes injected (PVI)

def run_simulation(mu_w):
    """
    Simulate the BL equation for a given water viscosity mu_w.

    Parameters
    ----------
    mu_w : float
        Water viscosity [cP].

    Returns
    -------
    x : ndarray (NX,)   cell-centre positions
    t : ndarray (NT,)   output times [PVI]
    S : ndarray (NX, NT) water saturation field
    """
    dx       = XMAX / NX
    dt_cfl   = 0.45 * dx / (U_TOT * 4.0)   # 4.0 is a safe upper bound on max|df/dS|
    t_out    = np.linspace(0.0, TMAX, NT)
    dt_store = t_out[1] - t_out[0]
    n_sub    = max(1, int(np.ceil(dt_store / dt_cfl)))
    dt       = dt_store / n_sub

    x = np.linspace(0.5 * dx, XMAX - 0.5 * dx, NX)
    S = np.full(NX, S_WC)            # initial: connate water everywhere
    S_out = np.zeros((NX, NT))
    S_out[:, 0] = S.copy()

    for k in range(1, NT):
        for _ in range(n_sub):
            fw            = fractional_flow(S, mu_w)
            flux_left     = np.empty(NX)
            flux_left[0]  = fractional_flow(np.array([1.0 - S_OR]), mu_w)[0]  # injection BC
            flux_left[1:] = fw[:-1]                                             # upwind
            S = np.clip(S - (U_TOT / dx) * (fw - flux_left) * dt,
                        S_WC, 1.0 - S_OR)
        S_out[:, k] = S.copy()

    return x, t_out, S_out

---
## 1. Fractional Flow Analysis

The shape of $f_w(S_w)$ and the Welge tangent point $S_w^*$ determine how fast and sharply
the flood front advances. Increasing $\mu_w$ (more viscous water, unfavourable mobility ratio)
shifts the shock to higher saturations and steeper fronts.

![Fractional Flow](screenshots/fractional_flow.png)


In [ ]:
MU_LIST = [0.5, 1.0, 2.0, 5.0]
COLORS  = ['#00ccff', '#99ff66', '#ffaa00', '#ff4444']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Fractional Flow & Relative Permeability', fontsize=13, fontweight='bold')

# ── Panel 1: f_w curves with Welge tangent ───────────────────────────────────
ax = axes[0]; _dark(ax)
sw_range = np.linspace(S_WC, 1 - S_OR, 300)
for mu_w, col in zip(MU_LIST, COLORS):
    fw = fractional_flow(sw_range, mu_w)
    ax.plot(sw_range, fw, color=col, linewidth=2, label=rf'$\mu_w$ = {mu_w:.1f} cP')
    sw_sh = shock_saturation(mu_w)
    fw_sh = fractional_flow(np.array([sw_sh]), mu_w)[0]
    ax.plot([S_WC, sw_sh], [0, fw_sh], '--', color=col, linewidth=1.2, alpha=0.7)
    ax.scatter([sw_sh], [fw_sh], color=col, s=35, zorder=5)
ax.set_xlabel(r'$S_w$'); ax.set_ylabel(r'$f_w$')
ax.set_title('Fractional flow curves\n(dashed = Welge tangent)', color='white')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=8, facecolor='#222', labelcolor='white', edgecolor='#555')

# ── Panel 2: Corey k_r curves ─────────────────────────────────────────────────
ax = axes[1]; _dark(ax)
sw_all = np.linspace(0, 1, 300)
krw, kro = rel_perms(sw_all)
ax.plot(sw_all, krw, color='#00ccff', linewidth=2, label=r'$k_{rw}$')
ax.plot(sw_all, kro, color='#ff8844', linewidth=2, label=r'$k_{ro}$')
ax.axvline(S_WC,     color='#777', linestyle=':', linewidth=1, label=r'$S_{wc}$')
ax.axvline(1 - S_OR, color='#aaa', linestyle=':', linewidth=1, label=r'$1-S_{or}$')
ax.set_xlabel(r'$S_w$'); ax.set_ylabel(r'$k_r$')
ax.set_title('Corey relative permeabilities', color='white')
ax.legend(fontsize=8, facecolor='#222', labelcolor='white', edgecolor='#555')

# ── Panel 3: shock saturation vs viscosity ────────────────────────────────────
ax = axes[2]; _dark(ax)
mu_range  = np.linspace(0.1, 10, 120)
sw_sh_arr = [shock_saturation(m) for m in mu_range]
ax.plot(mu_range, sw_sh_arr, color='#99ff66', linewidth=2)
ax.axvline(MU_O, color='#aaa', linestyle='--', linewidth=1,
           label=rf'$\mu_o$ = {MU_O:.0f} cP')
ax.set_xlabel(r'$\mu_w$ (cP)'); ax.set_ylabel(r'$S_w^*$ (shock saturation)')
ax.set_title('Shock saturation vs water viscosity', color='white')
ax.legend(fontsize=9, facecolor='#222', labelcolor='white', edgecolor='#555')

plt.tight_layout()
plt.show()

---
## 2. Saturation Front Evolution

Run the simulator at $\mu_w = 1\,\text{cP}$ and visualise the result as
time-snapshot profiles $S_w(x)$ and a space–time heatmap.

![Saturation Evolution](screenshots/saturation_evolution.png)


In [ ]:
x, t, S = run_simulation(mu_w=1.0)
norm = plt.Normalize(S_WC, 1 - S_OR)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle(r'Saturation evolution  ($\mu_w = 1.0$ cP)', fontsize=13, fontweight='bold')

# ── Panel 1: profiles at multiple times ──────────────────────────────────────
ax = axes[0]; _dark(ax)
for t_val in [0.1, 0.25, 0.5, 0.75, 1.0]:
    t_idx = int(np.argmin(np.abs(t - t_val)))
    ax.plot(x, S[:, t_idx], color=CMAP(t_val), linewidth=2.0,
            label=f't = {t_val:.2f} PVI')
ax.axhline(S_WC,     color='#555', linewidth=0.8, linestyle='--')
ax.axhline(1 - S_OR, color='#888', linewidth=0.8, linestyle='--')
ax.set_xlabel(r'$x/L$'); ax.set_ylabel(r'$S_w$')
ax.set_title('Saturation profiles at multiple times', color='white')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(fontsize=8.5, facecolor='#222', labelcolor='white', edgecolor='#555',
          bbox_to_anchor=(1.01, 1), loc='upper left')

# ── Panel 2: space-time heatmap ───────────────────────────────────────────────
ax = axes[1]; _dark(ax)
img  = ax.imshow(S, aspect='auto', origin='lower', cmap=CMAP,
                 extent=[0, TMAX, 0, XMAX], vmin=S_WC, vmax=1 - S_OR)
cbar = fig.colorbar(img, ax=ax, label=r'$S_w$')
cbar.ax.yaxis.label.set_color('white')
cbar.ax.tick_params(colors='white')
ax.set_xlabel('Time (PVI)'); ax.set_ylabel(r'$x/L$')
ax.set_title('Space\u2013time saturation map', color='white')

plt.tight_layout()
plt.show()

---
## 3. Effect of Mobility Ratio

The **mobility ratio** $M = (k_{rw}/\mu_w)\,/\,(k_{ro}/\mu_o)$ governs displacement efficiency:

- $M < 1$ — **favourable**: stable front, efficient sweep.
- $M > 1$ — **unfavourable**: viscous fingering tendency, earlier breakthrough.

Profiles below are shown at $t = 0.5$ PVI for four $\mu_w$ values.
The dashed red line is the analytical shock-front position from the Welge construction.

![Viscosity Comparison](screenshots/viscosity_comparison.png)


In [ ]:
T_FIXED = 0.5

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle(f'Saturation profiles at t = {T_FIXED} PVI \u2014 varying water viscosity',
             fontsize=13, fontweight='bold')

for ax, mu_w, col in zip(axes, MU_LIST, COLORS):
    _dark(ax)
    xv, tv, Sv = run_simulation(mu_w)
    t_idx = int(np.argmin(np.abs(tv - T_FIXED)))
    sw_sl = Sv[:, t_idx]

    # gradient colour line
    pts  = np.array([xv, sw_sl]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc   = LineCollection(segs, cmap=CMAP, norm=norm, linewidth=2.5)
    lc.set_array(sw_sl[:-1])
    ax.add_collection(lc)
    ax.fill_between(xv, S_WC, sw_sl, alpha=0.2, color=col)

    # analytical shock position
    sw_sh = shock_saturation(mu_w)
    fw_sh = fractional_flow(np.array([sw_sh]), mu_w)[0]
    x_fr  = min(fw_sh / (sw_sh - S_WC) * T_FIXED, XMAX)
    ax.axvline(x_fr, color='red', linewidth=1.4, linestyle=':',
               label=f'front x={x_fr:.2f}')

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel(r'$x/L$')
    ax.set_title(rf'$\mu_w$ = {mu_w:.1f} cP', color=col)
    ax.legend(fontsize=7.5, facecolor='#222', labelcolor='white', edgecolor='#555')

axes[0].set_ylabel(r'$S_w$')
plt.tight_layout()
plt.show()

---
## 4. Interactive Explorer

Three sliders let you explore the saturation field in real time:

| Slider | Effect |
|--------|--------|
| **Time (PVI)** | Saturation profile $S_w(x)$ at that time, with front marker |
| **x / L** | Saturation history $S_w(t)$ at that position |
| **\u03bc_w (cP)** | Reruns the simulation with the new water viscosity |

Requires `ipywidgets`:
```
pip install ipywidgets
```


In [ ]:
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print('ipywidgets not found.  Install with:  pip install ipywidgets')

if HAS_WIDGETS:
    _nb_cache = {}

    def _interactive_bl(mu_w=1.0, t_val=0.5, x_val=0.5):
        if _nb_cache.get('mu_w') != mu_w:
            xv, tv, Sv = run_simulation(mu_w)
            _nb_cache.update(mu_w=mu_w, x=xv, t=tv, S=Sv)
        xv = _nb_cache['x']; tv = _nb_cache['t']; Sv = _nb_cache['S']

        t_idx  = int(np.argmin(np.abs(tv - t_val)))
        x_idx  = int(np.argmin(np.abs(xv - x_val)))
        norm_l = plt.Normalize(S_WC, 1 - S_OR)

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        fig.suptitle(rf'Interactive BL Explorer  ($\mu_w$ = {mu_w:.2f} cP)',
                     fontsize=12, fontweight='bold')

        # ── time-slice profile ────────────────────────────────────────────────
        ax = axes[0]; _dark(ax)
        sw_t = Sv[:, t_idx]
        pts  = np.array([xv, sw_t]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc   = LineCollection(segs, cmap=CMAP, norm=norm_l, linewidth=2.5)
        lc.set_array(sw_t[:-1]); ax.add_collection(lc)
        ax.fill_between(xv, S_WC, sw_t, alpha=0.2, color='#00ffff')
        sw_sh = shock_saturation(mu_w)
        fw_sh = fractional_flow(np.array([sw_sh]), mu_w)[0]
        x_fr  = min(fw_sh / (sw_sh - S_WC) * tv[t_idx], XMAX)
        ax.axvline(x_fr, color='red', linewidth=1.4, linestyle=':',
                   label=f'front x={x_fr:.3f}')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel(r'$x/L$'); ax.set_ylabel(r'$S_w$')
        ax.set_title(f'Profile  t = {tv[t_idx]:.3f} PVI', color='#00ffff')
        ax.legend(fontsize=8, facecolor='#222', labelcolor='white', edgecolor='#555')

        # ── space-slice history ───────────────────────────────────────────────
        ax = axes[1]; _dark(ax)
        sw_x = Sv[x_idx, :]
        pts  = np.array([tv, sw_x]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc2  = LineCollection(segs, cmap=CMAP, norm=norm_l, linewidth=2.5)
        lc2.set_array(sw_x[:-1]); ax.add_collection(lc2)
        ax.fill_between(tv, S_WC, sw_x, alpha=0.2, color='#ffaa00')
        ax.axvline(tv[t_idx], color='#00ffff', linewidth=1.0, linestyle='--',
                   label=f't = {tv[t_idx]:.3f}')
        ax.set_xlim(0, TMAX); ax.set_ylim(0, 1)
        ax.set_xlabel('Time (PVI)'); ax.set_ylabel(r'$S_w$')
        ax.set_title(f'History  x = {xv[x_idx]:.3f} L', color='#ffaa00')
        ax.legend(fontsize=8, facecolor='#222', labelcolor='white', edgecolor='#555')

        # ── fractional-flow panel ─────────────────────────────────────────────
        ax = axes[2]; _dark(ax)
        sw_a = np.linspace(S_WC, 1 - S_OR, 300)
        fw_a = fractional_flow(sw_a, mu_w)
        ax.plot(sw_a, fw_a, color='#99ff66', linewidth=2)
        ax.plot([S_WC, sw_sh], [0, fw_sh], '--', color='#ff4444', linewidth=1.5,
                label=rf'Shock $S_w^*$ = {sw_sh:.2f}')
        ax.scatter([sw_sh], [fw_sh], color='red', zorder=5, s=40)
        ax.set_xlabel(r'$S_w$'); ax.set_ylabel(r'$f_w$')
        ax.set_title(rf'Fractional flow  ($\mu_w$ = {mu_w:.2f} cP)', color='#99ff66')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.legend(fontsize=8, facecolor='#222', labelcolor='white', edgecolor='#555')

        plt.tight_layout()
        plt.show()

    widgets.interact(
        _interactive_bl,
        mu_w  = widgets.FloatSlider(value=1.0, min=0.2, max=10.0, step=0.1,
                                    description='mu_w (cP)',
                                    style={'description_width': '80px'}),
        t_val = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                    description='Time (PVI)',
                                    style={'description_width': '80px'}),
        x_val = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.01,
                                    description='x / L',
                                    style={'description_width': '80px'}),
    )